# 03 - Exploración

Exploración de la capa plata (numeral 4 del documento). Todas las agregaciones se hacen en Spark y a pandas solo pasan los resultados, que ya son pequeños.

Lo que hay: 1 arrestos por año, 2 tasa de arrestos por borough, 3 gravedad de los arrestos, 4 delitos más frecuentes, 5 choques por mes, 6 choques graves por hora y día, 7 % de choques graves por borough, 8 factores contribuyentes, 9 tipos de vehículo, 10 pobreza, 11 educación, 12 SAT y 13 un panel que junta todo.

Hay que tener en cuenta que los arrestos miden la actividad de la policía, no cuánto crimen hay.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.ticker import FuncFormatter, PercentFormatter
from pyspark.sql import functions as F

from codigos import ORDEN_DIAS, POBREZA_EDUCACION
from config import SILVER, sesion
from estilo import BOROUGHS, CATEGORICA, COLOR_BOROUGH, SECUENCIAL, TINTA_SECUNDARIA, guardar, miles

spark = sesion("03_exploracion")
arrestos = spark.read.parquet(f"{SILVER}/arrestos").cache()
choques = spark.read.parquet(f"{SILVER}/choques").cache()
vehiculos = spark.read.parquet(f"{SILVER}/vehiculos")
pobreza = spark.read.parquet(f"{SILVER}/pobreza")
sat = spark.read.parquet(f"{SILVER}/sat")

ANIOS_COMPLETOS = list(range(2019, 2026))   # 2026 es parcial
fin_choques = choques.agg(F.max("fecha")).first()[0]
print(f"Arrestos: {arrestos.count():,} | Choques: {choques.count():,} (hasta {fin_choques}) | Vehículos: {vehiculos.count():,}")

def fmt_miles(ax, eje="y"):
    (ax.yaxis if eje == "y" else ax.xaxis).set_major_formatter(FuncFormatter(miles))

La población de cada borough es la suma de `PWGTP` en la tabla de pobreza (2018). Se usa la misma para todos los años.

In [ ]:
poblacion = pobreza.groupBy("borough").agg(F.sum("peso_persona").alias("poblacion")).toPandas().set_index("borough").reindex(BOROUGHS)
poblacion.assign(poblacion=lambda d: d["poblacion"].map(miles))

## 1. Arrestos por año

In [ ]:
por_anio = arrestos.groupBy("anio").count().orderBy("anio").toPandas()
fig, ax = plt.subplots(figsize=(8, 3.8))
colores = [CATEGORICA[0] if a <= 2025 else SECUENCIAL[1] for a in por_anio["anio"]]
ax.bar(por_anio["anio"].astype(str), por_anio["count"], color=colores, width=0.7)
for x, (a, v) in enumerate(zip(por_anio["anio"], por_anio["count"])):
    ax.text(x, v, miles(v) + ("\n(ene–jun)" if a == 2026 else ""), ha="center", va="bottom", fontsize=8, color=TINTA_SECUNDARIA)
fmt_miles(ax); ax.grid(axis="x", visible=False)
ax.set_title("Arrestos en Nueva York por año (2026: primer semestre)", pad=16)
guardar(fig, "03_01_arrestos_por_anio"); plt.show()

2026 va en un color más claro porque solo tiene seis meses.

## 2. Tasa de arrestos por 100.000 habitantes
Es el promedio por año entre 2019 y 2025.

In [ ]:
arr_b = (arrestos.filter(F.col("anio").isin(ANIOS_COMPLETOS) & F.col("borough").isNotNull())
         .groupBy("borough").count().toPandas().set_index("borough").reindex(BOROUGHS))
tasa_arr = (arr_b["count"] / len(ANIOS_COMPLETOS) / poblacion["poblacion"] * 1e5).round(0)

fig, ax = plt.subplots(figsize=(7, 3.8))
ax.bar(tasa_arr.index, tasa_arr.values, color=[COLOR_BOROUGH[b] for b in tasa_arr.index], width=0.65)
for x, v in enumerate(tasa_arr.values):
    ax.text(x, v, miles(v), ha="center", va="bottom", fontsize=9, color=TINTA_SECUNDARIA)
fmt_miles(ax); ax.grid(axis="x", visible=False)
ax.set_ylabel("arrestos por 100.000 hab. al año")
ax.set_title("Tasa anual de arrestos por borough (promedio 2019–2025)", pad=12)
guardar(fig, "03_02_tasa_arrestos_borough"); plt.show()

Manhattan hay que leerlo con cuidado: mucha gente que trabaja ahí o que está de visita no vive ahí, así que la tasa por residente puede salir inflada.

## 3. Arrestos por gravedad y borough
Cada barra suma 100 %.

In [ ]:
ORDEN_GRAV = ["Felonía", "Delito menor", "Infracción (violation)", "Otro o sin dato"]
COLOR_GRAV = dict(zip(ORDEN_GRAV, [CATEGORICA[0], CATEGORICA[1], CATEGORICA[2], "#b8b7b1"]))
grav_col = F.when(F.col("gravedad").isin("Otro / no documentado", "Sin dato"), "Otro o sin dato").otherwise(F.col("gravedad"))
grav = (arrestos.filter(F.col("borough").isNotNull()).withColumn("gravedad", grav_col)
        .groupBy("borough", "gravedad").count().toPandas()
        .pivot(index="borough", columns="gravedad", values="count").reindex(index=BOROUGHS, columns=ORDEN_GRAV).fillna(0))
grav_pct = grav.div(grav.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(8, 3.6))
izq = np.zeros(len(grav_pct))
for cat in ORDEN_GRAV:
    vals = grav_pct[cat].values
    ax.barh(grav_pct.index, vals, left=izq, color=COLOR_GRAV[cat], label=cat, height=0.6, edgecolor="white", linewidth=1)
    for y, (l, v) in enumerate(zip(izq, vals)):
        if v >= 8:
            ax.text(l + v / 2, y, f"{v:.0f} %", ha="center", va="center", fontsize=8, color="white")
    izq += vals
ax.xaxis.set_major_formatter(PercentFormatter(decimals=0)); ax.set_xlim(0, 100)
ax.invert_yaxis(); ax.grid(axis="y", visible=False)
ax.legend(loc="lower left", bbox_to_anchor=(0, 1.0), ncol=4, fontsize=8)
ax.set_title("Arrestos por nivel de gravedad en cada borough (2019 – jun 2026)", pad=26)
guardar(fig, "03_03_gravedad_borough"); plt.show()
grav_pct.round(1)

## 4. Los 10 tipos de delito más frecuentes

In [ ]:
top_delitos = (arrestos.groupBy("ofns_desc").count().orderBy(F.desc("count")).limit(10).toPandas()
               .assign(pct=lambda d: (100 * d["count"] / arrestos.count()).round(1)))
top_delitos.to_csv("../figuras/tabla_03_top_delitos.csv", index=False)
top_delitos.rename(columns={"ofns_desc": "tipo de delito (OFNS_DESC)", "count": "arrestos", "pct": "% del total"})

## 5. Choques por mes: total y con heridos o muertos

In [ ]:
mensual = (choques.groupBy("anio", "mes").agg(F.count("*").alias("total"), F.sum(F.col("grave").cast("int")).alias("graves"))
           .toPandas().assign(fecha=lambda d: pd.to_datetime(dict(year=d.anio, month=d.mes, day=1))).sort_values("fecha"))
fig, ax = plt.subplots(figsize=(10, 3.8))
ax.plot(mensual["fecha"], mensual["total"], color=CATEGORICA[0], label="Todos los choques")
ax.plot(mensual["fecha"], mensual["graves"], color=CATEGORICA[1], label="Con heridos o muertos")
for serie, color in [("total", CATEGORICA[0]), ("graves", CATEGORICA[1])]:
    ult = mensual.iloc[-1]
    ax.annotate(miles(ult[serie]), (ult["fecha"], ult[serie]), xytext=(4, 0), textcoords="offset points",
                fontsize=8, color=TINTA_SECUNDARIA, va="center")
fmt_miles(ax); ax.set_ylim(bottom=0)
ax.legend(loc="lower left", bbox_to_anchor=(0, 1.0), ncol=2, fontsize=8)
MESES = ["ene", "feb", "mar", "abr", "may", "jun", "jul", "ago", "sep", "oct", "nov", "dic"]
ax.set_title(f"Choques por mes en Nueva York (ene 2019 – {MESES[fin_choques.month - 1]} {fin_choques.year})", pad=24)
guardar(fig, "03_05_choques_por_mes"); plt.show()

El espacio entre las dos líneas son los choques sin heridos.

## 6. Choques graves por hora y día

In [ ]:
hm = (choques.filter(F.col("grave") & F.col("hora").isNotNull()).groupBy("dia_semana", "hora").count().toPandas()
      .pivot(index="dia_semana", columns="hora", values="count").reindex(index=ORDEN_DIAS, columns=range(24)).fillna(0))
cmap = LinearSegmentedColormap.from_list("azules", SECUENCIAL)
fig, ax = plt.subplots(figsize=(11, 3.6))
im = ax.imshow(hm.values, aspect="auto", cmap=cmap)
ax.set_xticks(range(24)); ax.set_xticklabels(range(24), fontsize=8)
ax.set_yticks(range(7)); ax.set_yticklabels(hm.index, fontsize=8)
ax.set_xlabel("hora del día"); ax.grid(False)
cb = fig.colorbar(im, ax=ax, pad=0.01); cb.set_label("choques graves", fontsize=8); cb.ax.tick_params(labelsize=7)
ax.set_title("Choques con heridos o muertos por hora y día de la semana (2019 – 2026)")
guardar(fig, "03_06_mapa_calor_hora_dia"); plt.show()
hm.astype(int).to_csv("../figuras/tabla_03_hora_dia.csv")

Entre más oscuro, más choques. Los números exactos quedan en `figuras/tabla_03_hora_dia.csv`.

## 7. % de choques graves por borough
Ya incluye los choques a los que se les recuperó el borough con las coordenadas (notebook 05).

In [ ]:
pct_grave = (choques.groupBy(F.coalesce("borough", F.lit("Sin dato")).alias("borough"))
             .agg(F.count("*").alias("choques"), F.avg(F.col("grave").cast("int")).alias("pct_graves")).toPandas()
             .set_index("borough").reindex(BOROUGHS + ["Sin dato"]).dropna(how="all"))
pct_grave["pct_graves"] *= 100
fig, ax = plt.subplots(figsize=(7, 3.8))
ax.bar(pct_grave.index, pct_grave["pct_graves"], width=0.65,
       color=[COLOR_BOROUGH.get(b, "#b8b7b1") for b in pct_grave.index])
for x, v in enumerate(pct_grave["pct_graves"]):
    ax.text(x, v, f"{v:.1f} %", ha="center", va="bottom", fontsize=9, color=TINTA_SECUNDARIA)
ax.yaxis.set_major_formatter(PercentFormatter(decimals=0)); ax.grid(axis="x", visible=False)
ax.tick_params(axis="x", labelsize=8)
ax.set_title("Porcentaje de choques con heridos o muertos, por borough", pad=12)
guardar(fig, "03_07_pct_graves_borough"); plt.show()
pct_grave.assign(choques=lambda d: d["choques"].map(miles), pct_graves=lambda d: d["pct_graves"].round(1))

## 8. Factores contribuyentes
Se usa el factor del vehículo 1 y se quita `Unspecified`.

In [ ]:
total_ch = choques.count()
unspec = choques.filter(F.col("factor_1") == "Unspecified").count()
factores = (choques.filter(F.col("factor_1").isNotNull() & (F.col("factor_1") != "Unspecified"))
            .groupBy("factor_1").count().orderBy(F.desc("count")).limit(10).toPandas().sort_values("count"))
fig, ax = plt.subplots(figsize=(8, 4.2))
ax.barh(factores["factor_1"], factores["count"], color=CATEGORICA[0], height=0.65)
for y, v in zip(factores["factor_1"], factores["count"]):
    ax.text(v, y, " " + miles(v), va="center", fontsize=8, color=TINTA_SECUNDARIA)
fmt_miles(ax, "x"); ax.grid(axis="y", visible=False); ax.tick_params(axis="y", labelsize=8)
ax.set_title(f"Los 10 factores contribuyentes más frecuentes\n(sin contar 'Unspecified', que es el {100 * unspec / total_ch:.1f} % de los choques)")
guardar(fig, "03_08_factores"); plt.show()

## 9. Tipos de vehículo más frecuentes

In [ ]:
total_v = vehiculos.filter(F.col("tiene_choque")).count()
sin_tipo = vehiculos.filter(F.col("tiene_choque") & (F.col("tipo_vehiculo").isNull() | (F.trim("tipo_vehiculo") == ""))).count()
print(f"Vehículos sin tipo registrado: {sin_tipo:,} ({100 * sin_tipo / total_v:.1f} %), no entran en el ranking")
tipos = (vehiculos.filter(F.col("tiene_choque") & F.col("tipo_vehiculo").isNotNull() & (F.trim("tipo_vehiculo") != ""))
         .groupBy("tipo_vehiculo").count().orderBy(F.desc("count")).limit(10).toPandas()
         .assign(pct=lambda d: (100 * d["count"] / total_v).round(1)))
tipos.to_csv("../figuras/tabla_03_tipos_vehiculo.csv", index=False)
tipos.rename(columns={"tipo_vehiculo": "tipo de vehículo", "count": "vehículos", "pct": "% del total"})

El tipo de vehículo es texto libre y un mismo tipo aparece escrito de varias formas. Eso se normaliza más adelante.

## 10. Tasa de pobreza por borough
Ponderada con `PWGTP`.

In [ ]:
pob = (pobreza.groupBy("borough")
       .agg((F.sum(F.when(F.col("en_pobreza"), F.col("peso_persona")).otherwise(0)) / F.sum("peso_persona") * 100).alias("tasa_pobreza"))
       .toPandas().set_index("borough").reindex(BOROUGHS))
fig, ax = plt.subplots(figsize=(7, 3.8))
ax.bar(pob.index, pob["tasa_pobreza"], color=[COLOR_BOROUGH[b] for b in pob.index], width=0.65)
for x, v in enumerate(pob["tasa_pobreza"]):
    ax.text(x, v, f"{v:.1f} %", ha="center", va="bottom", fontsize=9, color=TINTA_SECUNDARIA)
ax.yaxis.set_major_formatter(PercentFormatter(decimals=0)); ax.grid(axis="x", visible=False)
ax.set_title("Tasa de pobreza (medida NYCgov) por borough, 2018", pad=12)
guardar(fig, "03_10_pobreza_borough"); plt.show()

## 11. Nivel educativo por borough
Solo mayores de 18, ponderado con `PWGTP`.

In [ ]:
ORDEN_EDU = [POBREZA_EDUCACION[k] for k in sorted(POBREZA_EDUCACION)]
edu = (pobreza.filter((F.col("edad") >= 18) & F.col("educacion").isNotNull())
       .groupBy("borough", "educacion").agg(F.sum("peso_persona").alias("personas")).toPandas()
       .pivot(index="borough", columns="educacion", values="personas").reindex(index=BOROUGHS, columns=ORDEN_EDU).fillna(0))
edu_pct = edu.div(edu.sum(axis=1), axis=0) * 100
colores_edu = [SECUENCIAL[1], SECUENCIAL[2], SECUENCIAL[3], SECUENCIAL[5]]
fig, ax = plt.subplots(figsize=(8, 3.6))
izq = np.zeros(len(edu_pct))
for cat, color in zip(ORDEN_EDU, colores_edu):
    vals = edu_pct[cat].values
    ax.barh(edu_pct.index, vals, left=izq, color=color, label=cat, height=0.6, edgecolor="white", linewidth=1)
    for y, (l, v) in enumerate(zip(izq, vals)):
        if v >= 8:
            ax.text(l + v / 2, y, f"{v:.0f} %", ha="center", va="center", fontsize=8,
                    color="white" if color in (SECUENCIAL[3], SECUENCIAL[5]) else "#0b0b0b")
    izq += vals
ax.xaxis.set_major_formatter(PercentFormatter(decimals=0)); ax.set_xlim(0, 100)
ax.invert_yaxis(); ax.grid(axis="y", visible=False)
ax.legend(loc="lower left", bbox_to_anchor=(0, 1.0), ncol=4, fontsize=8)
ax.set_title("Nivel educativo de los adultos por borough, 2018", pad=26)
guardar(fig, "03_11_educacion_borough"); plt.show()

## 12. SAT por borough (2012)
Promedio de los colegios de cada borough, dejando por fuera los puntajes suprimidos.

In [ ]:
sat_b = (sat.groupBy("borough").agg(
            F.count("*").alias("colegios"),
            F.sum(F.col("suprimido").cast("int")).alias("suprimidos"),
            F.round(F.avg("lectura"), 0).alias("lectura"),
            F.round(F.avg("matematicas"), 0).alias("matemáticas"),
            F.round(F.avg("escritura"), 0).alias("escritura"),
            F.round(F.avg("total"), 0).alias("total (máx. 2400)"))
         .toPandas().set_index("borough").reindex(BOROUGHS))
sat_b.to_csv("../figuras/tabla_03_sat_borough.csv")
sat_b

## 13. Panel comparativo
Junta los indicadores de cada borough en una sola tabla. Nos sirve para las preguntas 2 y 7.

In [ ]:
graves_b = (choques.filter(F.col("anio").isin(ANIOS_COMPLETOS) & F.col("grave") & F.col("borough").isNotNull())
            .groupBy("borough").count().toPandas().set_index("borough").reindex(BOROUGHS)["count"])
panel = pd.DataFrame({
    "población (ACS 2018)": poblacion["poblacion"].map(miles),
    "pobreza %": pob["tasa_pobreza"].round(1),
    "adultos con pregrado o más %": edu_pct["Pregrado o más"].round(1),
    "SAT total promedio": sat_b["total (máx. 2400)"],
    "arrestos / 100 mil hab. / año": tasa_arr.map(miles),
    "choques graves / 100 mil hab. / año": (graves_b / len(ANIOS_COMPLETOS) / poblacion["poblacion"] * 1e5).round(0).map(miles),
})
panel.to_csv("../figuras/tabla_03_panel_borough.csv")
panel

## Conclusiones

- Quedaron 13 elementos de análisis, todos agregados en Spark.
- Comparamos tasas y porcentajes y no conteos, porque los boroughs tienen poblaciones muy distintas.
- Pobreza y educación están ponderadas con `PWGTP`.
- Limitaciones: la población es de 2018 y se usa para todo el periodo, los arrestos reflejan actividad policial y 2026 está incompleto.